In [8]:
# task1.1: Define the stack and data structure
class Stack:
    def __init__(self):
        # 初始化一个原生的 Python 列表来存储数据
        self.items = []

    def isEmpty(self):
        # 如果列表长度为0，返回 True，否则返回 False
        return len(self.items) == 0

    def push(self, item):
        # 将元素添加到列表末尾（栈顶）
        self.items.append(item)

    def pop(self):
        # 如果栈不为空，移除并返回列表末尾（栈顶）的元素
        if not self.isEmpty():
            return self.items.pop()
        return None

    def top(self):
        # 如果栈不为空，返回列表末尾（栈顶）的元素，但不移除它
        if not self.isEmpty():
            return self.items[-1]
        return None

In [9]:
# task1.2: Define the cell class
class Cell:
    def __init__(self, row_index, col_index):
        # 初始化坐标
        self.row = row_index
        self.col = col_index
        
        # 初始状态下四面都有墙
        self.N = True
        self.S = True
        self.E = True
        self.W = True
        
        # 记录是否被访问过
        self.visited = False

    def coordinate(self):
        # 返回一个包含坐标的元组
        return (self.row, self.col)

    def breakWall(self, direction):
        # 根据传入的方向打破对应的墙壁
        if direction == 'N':
            self.N = False
        elif direction == 'S':
            self.S = False
        elif direction == 'E':
            self.E = False
        elif direction == 'W':
            self.W = False

In [10]:
# task 1.3: Initialize the maze and the draw_maze function
# 初始化迷宫，用于测试 draw_maze
width = 10
height = 10
maze = [[Cell(r, c) for c in range(width)] for r in range(height)]

def draw_maze(maze, width, height):
    for r in range(height):
        # 1. 打印当前行所有格子的北墙 (North walls)
        for c in range(width):
            print('+', end='')
            if maze[r][c].N:
                print('--', end='')
            else:
                print('  ', end='')
        print('+') # 当前行北墙结束
        
        # 2. 打印当前行所有格子的西墙 (West walls)
        for c in range(width):
            if maze[r][c].W:
                print('|', end='')
            else:
                print(' ', end='')
            print('  ', end='') # 内部宽度为2的空格
            
        # 3. 打印当前行最右侧格子的东墙 (East wall of the last cell)
        if maze[r][width-1].E:
            print('|')
        else:
            print(' ')
            
    # 4. 打印迷宫最底部的南墙 (South walls of the last row)
    for c in range(width):
        print('+', end='')
        if maze[height-1][c].S:
            print('--', end='')
        else:
            print('  ', end='')
    print('+')

In [11]:
# task 1.4: Implement the getNeighbors and breakWallBetweenTwoCell function
def getNeighbors(maze, cell, width, height):
    neighbors = []
    r, c = cell.coordinate()
    
    # 检查北边 (North): row - 1
    if r > 0 and not maze[r-1][c].visited:
        neighbors.append(maze[r-1][c])
    # 检查南边 (South): row + 1
    if r < height - 1 and not maze[r+1][c].visited:
        neighbors.append(maze[r+1][c])
    # 检查西边 (West): col - 1
    if c > 0 and not maze[r][c-1].visited:
        neighbors.append(maze[r][c-1])
    # 检查东边 (East): col + 1
    if c < width - 1 and not maze[r][c+1].visited:
        neighbors.append(maze[r][c+1])
        
    return neighbors # 返回未访问邻居的列表

def breakWallBetweenTwoCells(u, v):
    u_r, u_c = u.coordinate()
    v_r, v_c = v.coordinate()
    
    # u 在 v 的下面 (u's North is v's South)
    if u_r > v_r:
        u.breakWall('N')
        v.breakWall('S')
    # u 在 v 的上面 (u's South is v's North)
    elif u_r < v_r:
        u.breakWall('S')
        v.breakWall('N')
    # u 在 v 的右边 (u's West is v's East)
    elif u_c > v_c:
        u.breakWall('W')
        v.breakWall('E')
    # u 在 v 的左边 (u's East is v's West)
    elif u_c < v_c:
        u.breakWall('E')
        v.breakWall('W')

In [12]:
# task 1.5: Implement the maze_generation function
import random

def maze_generation(width, height):
    # 初始化迷宫：创建一个 w x h 的网格，所有格子初始未访问，墙壁完好 
    maze = [[Cell(r, c) for c in range(width)] for r in range(height)]
    
    # 初始化 Stack
    stack = Stack()
    
    # 设定起点和终点
    start_cell = maze[0][0]
    
    # 标记入口格子为已访问，并压入栈中
    start_cell.visited = True
    stack.push(start_cell)
    
    # 只要栈不为空，就继续循环 
    while not stack.isEmpty():
        # 获取栈顶元素 
        current_cell = stack.top()
        
        # 获取未访问的邻居
        neighbors = getNeighbors(maze, current_cell, width, height)
        
        if len(neighbors) > 0: # 如果有未访问的邻居 
            # 随机选择一个未访问的邻居 
            chosen_neighbor = random.choice(neighbors)
            
            # 移除当前单元格和选定邻居之间的墙 
            breakWallBetweenTwoCells(current_cell, chosen_neighbor)
            
            # 标记选定的邻居为已访问 
            chosen_neighbor.visited = True
            
            # 将选定的邻居压入栈中 
            stack.push(chosen_neighbor)
        else: # 如果没有未访问的邻居 
            # 从栈中弹出当前单元格 (回溯) 
            stack.pop()
            
    # 返回生成的二维迷宫，供 Task 2 使用 
    return maze

In [13]:
# Define the LinkedList and data structure 
class Node:
    def __init__(self, coord):
        # 存储当前格子的坐标
        self.coord = coord
        
        # 使用四个方向替代传统的 next 引用
        self.N = None
        self.S = None
        self.E = None
        self.W = None

class LinkedList:
    def __init__(self):
        # 严格遵守要求：只维护一个 head 引用
        self.head = None
        
    def _get_next_node_and_direction(self, curr_node):
        # 辅助方法：找出一个节点的下一个连接节点和对应的方向
        if curr_node.N is not None: return curr_node.N, 'N'
        if curr_node.S is not None: return curr_node.S, 'S'
        if curr_node.E is not None: return curr_node.E, 'E'
        if curr_node.W is not None: return curr_node.W, 'W'
        return None, None

    def __str__(self):
        # 如果链表为空，返回空字符串
        if self.head is None:
            return ""
            
        # 拼接字符串
        result = f"({self.head.coord[0]},{self.head.coord[1]})"
        curr = self.head
        
        while True:
            next_node, direction = self._get_next_node_and_direction(curr)
            if next_node is None:
                break
            result += f"+-{direction}-+({next_node.coord[0]},{next_node.coord[1]})"
            curr = next_node
            
        return result

    def add(self, newNode):
        # 将新节点添加到链表尾部
        if self.head is None:
            self.head = newNode
            return
            
        # 从头节点开始遍历，找到最后一个节点
        curr = self.head
        while True:
            next_node, _ = self._get_next_node_and_direction(curr)
            if next_node is None:
                break
            curr = next_node
            
        # 比较坐标，判断新节点在当前节点的哪个方向
        r1, c1 = curr.coord
        r2, c2 = newNode.coord
        
        if r2 == r1 - 1:
            curr.N = newNode
        elif r2 == r1 + 1:
            curr.S = newNode
        elif c2 == c1 + 1:
            curr.E = newNode
        elif c2 == c1 - 1:
            curr.W = newNode

    def removeLastNode(self):
        # 遇到死胡同时，移除链表的最后一个节点
        if self.head is None:
            return
            
        next_node, _ = self._get_next_node_and_direction(self.head)
        # 如果链表只有一个节点
        if next_node is None:
            self.head = None
            return
            
        # 寻找倒数第二个节点 (prev)
        curr = self.head
        prev = None
        direction_from_prev = None
        
        while True:
            next_node, direction = self._get_next_node_and_direction(curr)
            if next_node is None:
                break
            prev = curr
            direction_from_prev = direction
            curr = next_node
            
        # 断开倒数第二个节点与最后一个节点的连接
        if direction_from_prev == 'N': prev.N = None
        elif direction_from_prev == 'S': prev.S = None
        elif direction_from_prev == 'E': prev.E = None
        elif direction_from_prev == 'W': prev.W = None

In [14]:
# task 2.2:provide the code to resolve the maze
def maze_resolution(maze):
    height = len(maze)
    width = len(maze[0])
    target_coord = (height - 1, width - 1)
    
    # 初始化我们自己写的栈和链表
    stack = Stack()
    llist = LinkedList()
    
    # 使用一个集合记录在求解过程中已经访问过的坐标，防止走回头路
    visited = set()
    
    # 从入口 (0, 0) 开始
    start_coord = (0, 0)
    stack.push(start_coord)
    
    while not stack.isEmpty():
        curr_coord = stack.top()
        r, c = curr_coord
        
        # 如果是第一次走到这个格子，加入访问记录并放入链表中
        if curr_coord not in visited:
            visited.add(curr_coord)
            llist.add(Node(curr_coord))
            
        # 如果到达终点，直接结束搜索
        if curr_coord == target_coord:
            break
            
        cell = maze[r][c]
        next_step = None
        
        # 寻找下一个可以走且未被访问过的邻居（中间没有墙）
        # 优先级：南 -> 东 -> 北 -> 西
        if not cell.S and (r + 1, c) not in visited:
            next_step = (r + 1, c)
        elif not cell.E and (r, c + 1) not in visited:
            next_step = (r, c + 1)
        elif not cell.N and (r - 1, c) not in visited:
            next_step = (r - 1, c)
        elif not cell.W and (r, c - 1) not in visited:
            next_step = (r, c - 1)
            
        if next_step is not None:
            # 找到可以走的路，压入栈中继续深入
            stack.push(next_step)
        else:
            # 四个方向都走不通，说明是死胡同，触发回溯机制
            stack.pop()
            llist.removeLastNode()
            
    # 通过 print(llist) 在外部验证打印结果
    return llist

In [21]:
# 生成并打印一个小迷宫 (5x5)
print("生成的小迷宫 (5x5):")
small_width = 5
small_height = 5
small_maze = maze_generation(small_width, small_height)
draw_maze(small_maze, small_width, small_height)
solution_path = maze_resolution(my_maze)

# 打印出最终的链表路线
print(solution_path)

# 生成并打印一个大迷宫 (10x10)
print("生成的大迷宫 (10x10):")
large_width = 10
large_height = 10
large_maze = maze_generation(large_width, large_height)
draw_maze(large_maze, large_width, large_height)
solution_path = maze_resolution(my_maze)

# 打印出最终的链表路线
print(solution_path)

生成的小迷宫 (5x5):
+--+--+--+--+--+
|  |           |
+  +  +--+--+  +
|     |        |
+--+--+  +--+--+
|  |     |     |
+  +  +--+--+  +
|  |  |        |
+  +  +  +--+  +
|        |     |
+--+--+--+--+--+
(0,0)+-E-+(0,1)+-S-+(1,1)+-W-+(1,0)+-S-+(2,0)+-E-+(2,1)+-E-+(2,2)+-S-+(3,2)+-E-+(3,3)+-S-+(4,3)+-S-+(5,3)+-W-+(5,2)+-W-+(5,1)+-S-+(6,1)+-S-+(7,1)+-E-+(7,2)+-N-+(6,2)+-E-+(6,3)+-S-+(7,3)+-E-+(7,4)+-S-+(8,4)+-E-+(8,5)+-E-+(8,6)+-S-+(9,6)+-E-+(9,7)+-E-+(9,8)+-E-+(9,9)
生成的大迷宫 (10x10):
+--+--+--+--+--+--+--+--+--+--+
|     |              |        |
+--+  +--+--+  +--+  +  +--+  +
|  |        |  |  |        |  |
+  +--+--+  +  +  +--+--+--+  +
|        |  |     |        |  |
+--+--+  +  +--+--+  +--+  +  +
|        |     |     |  |     |
+  +--+--+--+  +  +--+  +--+  +
|  |        |     |  |        |
+  +  +--+  +--+--+  +  +--+--+
|  |  |  |        |     |     |
+  +  +  +--+--+  +  +--+  +  +
|     |        |  |  |     |  |
+  +--+--+  +  +  +  +--+  +  +
|  |     |  |  |  |  |     |  |
+  + 

Task 2.3

The maze_resolution function employs a Depth-First Search (DFS) algorithm combined with a backtracking mechanism to find a valid path from the entrance (0, 0) to the destination. It utilizes three main data structures: a custom Stack to manage the exploration path, a set to keep track of visited coordinates (preventing infinite loops), and a custom LinkedList to store the exact directional solution.

At each step, the algorithm examines the cell at the top of the stack. It checks the four adjacent directions (South, East, North, West) to find a neighbor that is accessible (no wall blocking the way) and has not been visited yet.

Moving Forward: If a valid unvisited neighbor is found, the algorithm moves to that cell. It adds the new coordinate to the visited set, pushes it onto the stack, and appends it to the LinkedList as part of the current path.


Backtracking: If there are no further routes to reach the destination from the current cell, it means we have encountered a dead-end node. To handle this, the algorithm triggers a backtracking technique. It pops the dead-end cell from the stack and calls removeLastNode() on the LinkedList to exclude it from the final solution path.

This process continuously explores deep into the maze and backtracks whenever it hits a dead end, repeating until the coordinates of the destination are successfully reached.